In [1]:
from google.cloud import bigquery
import os

PROJECT_ID = os.getenv("GCP_PROJECT_ID")
DATASET_ID = os.getenv("BQ_DATASET_ID")
client = bigquery.Client(project=PROJECT_ID, location="EU")

def run_query(title, sql):
    print(f"\n{'='*60}")
    print(f"  {title}")
    print(f"{'='*60}")
    df = client.query(sql).to_dataframe()
    print(df.to_string(index=False))
    return df

c:\Users\user\Documents\GitHub\tc-sql-Innovaciber\venv\lib\site-packages\google\api_core\_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.11) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [2]:
print(PROJECT_ID)
print(DATASET_ID)

project-b9801b85-0d7a-44a1-84e
datalearn_academy


In [3]:
# QUERY 1: Ingresos por mes 

q1 = f"""
SELECT
  FORMAT_TIMESTAMP('%Y-%m', o.order_date) AS mes,
  COUNT(DISTINCT o.order_id)              AS num_pedidos,
  ROUND(SUM(oi.unit_price * oi.quantity), 2) AS ingresos_totales
FROM `{PROJECT_ID}.{DATASET_ID}.orders`      AS o
JOIN `{PROJECT_ID}.{DATASET_ID}.order_items` AS oi
  ON o.order_id = oi.order_id
WHERE o.order_status = 'delivered'
GROUP BY mes
ORDER BY mes
"""

df_q1 = run_query("Q1 — Ingresos por mes", q1)


  Q1 — Ingresos por mes
    mes  num_pedidos  ingresos_totales
2025-05          164         176846.58
2025-06          144         140929.84
2025-07          154         156459.22
2025-08          159         167961.31
2025-09          160         171608.32
2025-10          162         174684.51
2025-11          161         159365.47
2025-12          191         190501.07
2026-01          160         164650.68
2026-02          187         206977.08
2026-03          153         157791.23
2026-04          164         155058.14
2026-05           41          47032.84


In [4]:
# QUERY 2: Top 10 productos más vendidos

q2 = f"""
SELECT
  p.product_name                             AS producto,
  p.brand                                    AS marca,
  SUM(oi.quantity)                           AS unidades_vendidas,
  ROUND(SUM(oi.unit_price * oi.quantity), 2) AS revenue_total
FROM `{PROJECT_ID}.{DATASET_ID}.order_items` AS oi
JOIN `{PROJECT_ID}.{DATASET_ID}.products`    AS p
  ON oi.product_id = p.product_id
GROUP BY producto, marca
ORDER BY unidades_vendidas DESC
LIMIT 10
"""

df_q2 = run_query("Q2 — Top 10 productos más vendidos", q2)


  Q2 — Top 10 productos más vendidos
producto                       marca  unidades_vendidas  revenue_total
   Movie      Reese, Smith and Ellis                270       54918.00
  Future   Rodriguez, Khan and Singh                220       29986.00
    View                Jones-Stuart                216       33222.96
Remember              Gilbert-Spears                212       35541.80
  Budget   York, Edwards and Edwards                210       18366.60
    Most               Wagner-Sexton                206       24371.86
     Red Pope, Williams and Williams                206       49479.14
 Discuss              Moore and Sons                205       48802.30
  Myself  Moon, Mcmahon and Robinson                203       30659.09
Possible             Walters-Parrish                197       40662.77


In [5]:
# QUERY 3: Pedidos y ticket medio por país 
q3 = f"""
SELECT
  cu.country                     AS pais,
  COUNT(DISTINCT cu.customer_id) AS total_clientes,
  COUNT(DISTINCT o.order_id)     AS total_pedidos,
  ROUND(AVG(pay.amount), 2)      AS ticket_medio
FROM `{PROJECT_ID}.{DATASET_ID}.customers` AS cu
LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.orders`   AS o
  ON cu.customer_id = o.customer_id
LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.payments` AS pay
  ON o.order_id = pay.order_id
GROUP BY pais
ORDER BY total_pedidos DESC
"""

df_q3 = run_query("Q3 — Pedidos y ticket medio por país", q3)


  Q3 — Pedidos y ticket medio por país
pais  total_clientes  total_pedidos  ticket_medio
  EC               8             41       1069.22
  MC               6             30       1033.85
  UG               4             27        988.76
  SO               8             27       1068.45
  PW               5             26       1019.63
  DK               6             26       1046.80
  GH               7             25       1002.90
  DZ               6             24        882.69
  LA               6             24       1123.70
  BB               4             24       1067.16
  LC               5             24       1223.58
  GA               5             23        940.60
  MN               5             22        991.65
  HT               5             21       1030.69
  KG               5             21       1094.72
  AO               4             21       1018.54
  BN               4             21       1013.47
  MV               6             21        921.91
  ID      

In [6]:
# QUERY 4: Tiempo medio de entrega por país
q4 = f"""
SELECT
  s.shipping_country AS pais,
  ROUND(AVG(
    TIMESTAMP_DIFF(s.delivery_date, o.order_date, DAY)
  ), 1)              AS dias_entrega_medio,
  COUNT(*)           AS pedidos_entregados
FROM `{PROJECT_ID}.{DATASET_ID}.shipments` AS s
JOIN `{PROJECT_ID}.{DATASET_ID}.orders`    AS o
  ON s.order_id = o.order_id
WHERE s.shipping_status = 'delivered'
GROUP BY pais
ORDER BY dias_entrega_medio ASC
"""

df_q4 = run_query("Q4 — Tiempo medio de entrega por país", q4)


  Q4 — Tiempo medio de entrega por país
pais  dias_entrega_medio  pedidos_entregados
  NA                 1.8                   6
  TN                 1.8                   4
  NI                 1.9                   7
  SB                 1.9                   9
  EE                 2.0                   5
  MU                 2.0                  10
  AL                 2.1                  13
  ME                 2.1                   8
  UG                 2.2                  11
  NZ                 2.2                   5
  KR                 2.2                   9
  AD                 2.2                   6
  BR                 2.2                  10
  ZW                 2.2                  14
  VN                 2.3                  11
  KE                 2.3                   6
  TJ                 2.3                   8
  AO                 2.4                   8
  RW                 2.4                  10
  IE                 2.4                   9
  YE          

In [7]:
# QUERY 5: Margen de beneficio por categoría 

q5 = f"""
SELECT
  cat.name                                                    AS categoria,
  ROUND(SUM(oi.unit_price * oi.quantity), 2)                  AS ingresos,
  ROUND(SUM(p.cost_price  * oi.quantity), 2)                  AS coste,
  ROUND(SUM((oi.unit_price - p.cost_price) * oi.quantity), 2) AS margen_bruto,
  ROUND(
    100.0 * SUM((oi.unit_price - p.cost_price) * oi.quantity)
    / NULLIF(SUM(oi.unit_price * oi.quantity), 0)
  , 2)                                                        AS margen_pct
FROM `{PROJECT_ID}.{DATASET_ID}.order_items` AS oi
JOIN `{PROJECT_ID}.{DATASET_ID}.products`    AS p
  ON oi.product_id = p.product_id
JOIN `{PROJECT_ID}.{DATASET_ID}.categories`  AS cat
  ON p.category_id = cat.category_id
GROUP BY categoria
ORDER BY margen_pct DESC
"""

df_q5 = run_query("Q5 — Margen de beneficio por categoría", q5)


  Q5 — Margen de beneficio por categoría
  categoria  ingresos     coste  margen_bruto  margen_pct
       Food 122545.24  59854.89      62690.35       51.16
       Home 119291.19  61069.56      58221.63       48.81
      Books 237105.20 123981.97     113123.23       47.71
     Beauty 135651.06  72074.74      63576.32       46.87
 Automotive 209244.42 114100.08      95144.34       45.47
Electronics 221635.36 123965.35      97670.01       44.07
     Garden 241845.56 135414.32     106431.24       44.01
       Toys 240774.69 136182.72     104591.97       43.44
   Clothing 187345.80 106589.19      80756.61       43.11
     Sports 354427.77 205712.27     148715.50       41.96
